In [1]:
import glob
import os
from pathlib import Path

import numpy as np
import tqdm
import xmltodict
import zarr
from bioio import BioImage
from multiview_stitcher import fusion
from multiview_stitcher import spatial_image_utils as si_utils
from ome_zarr import (
    OMEZarrHCSPlate,
    OMEZarrImage,
    OMEZarrMultiscale,
    OMEZarrScene,
)
from ome_zarr_models.v06.coordinate_transforms import (
    Axis,
    CoordinateSystem,
    CoordinateSystemIdentifier,
    Translation,
)

zarr.config.set({"codec_pipeline.path": "zarrs.ZarrsCodecPipeline"})

c:\Users\aifadmin\Documents\Github\autoSCAPE\.venv\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [2]:
# client = Client()
# client.dashboard_link

In [3]:
# viewer = napari.Viewer()

In [12]:
def find_in_nested_dict(data, key, value):
    """
    Recursively search for objects containing a specific key-value pair.
    Returns a list of all matching objects.
    """
    results = []

    def search(obj):
        if isinstance(obj, dict):
            if obj.get(key) == value:
                results.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)

    search(data)
    return results

## Convert overview images

This part of the notebook handles the conversion of brightfield data into a joint scene layout.

In [4]:
root = r'D:\UserData\Johannes\20260512\2026_05_12_10_26_00--20260507_Plate2_Experiment1001'
file_names = glob.glob(os.path.join(root, "TileScan 1", "**", "*ome.tif"), recursive=True) + glob.glob(os.path.join(root, "TileScan 5", "**", "*ome.tif"), recursive=True)
len(file_names)

96

In [ ]:
write_jobs = []
images = []
transforms = []
for filename in tqdm.tqdm(file_names):
	img_basename = Path(filename).stem.split(".")[0]
	outfile_oz = img_basename + ".ome.zarr"
	bf = BioImage(filename)


	if bf.ome_metadata.images[0].pixels.physical_size_z is None:
		size_z = 0
	else:
		size_z = bf.ome_metadata.images[0].pixels.physical_size_z

	scale = {
		"z": size_z,
		"y": bf.ome_metadata.images[0].pixels.physical_size_y,
		"x": bf.ome_metadata.images[0].pixels.physical_size_x
	}

	axes_units = {
		"z": "micrometer",
		"y": "micrometer",
		"x": "micrometer",
	}

	# parse tile metadata
	metadata_file = os.path.join(Path(filename).parent, "Metadata", f"{img_basename}.xlif")
	with open(metadata_file, 'rb') as f:
		content = f.read().decode('utf-8')
		metadata_dict = xmltodict.parse(content)
	tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

	# n_series= bf.series_count()
	n_series = len(bf.scenes)

	all_positions = []
	if n_series > 1:
		sims = []
		for idx in range(n_series):
			bf.set_scene(bf.scenes[idx])
			dask_image = bf.dask_data
			size_tile = np.asarray(dask_image.squeeze().shape) * np.asarray([scale["z"], scale["y"], scale["x"]])
			# dask_image = bf.to_dask(series=idx)
			ts_info = tile_scan_info[idx]

			t_z = float(ts_info["@PosZ"])
			t_y = float(ts_info["@PosY"])
			t_x = float(ts_info["@PosX"])

			all_positions.append(np.asarray([t_z, t_y, t_x]))

			sims.append(
				si_utils.get_sim_from_array(
					dask_image,
					dims=["t", "c", "z", "y", "x"],
					scale=scale,
					translation={"z": t_z, "y": t_y, "x": t_x},
					transform_key="stage_metadata",
					)
			)

		image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion, backend="cupy")
	else:
		# image = bf.to_dask(series=0)
		image = bf.dask_data
		size_tile = np.asarray(dask_image.squeeze()) * np.asarray([scale["z"], scale["y"], scale["x"]])
		t_z = float(tile_scan_info["@PosZ"])
		t_y = float(tile_scan_info["@PosY"])
		t_x = float(tile_scan_info["@PosX"])

		all_positions.append(np.asarray([t_z, t_y, t_x]))

	all_positions = np.stack(all_positions, axis=0)
	position_corner = np.min(all_positions, axis=0) - size_tile / 2

	array = image.squeeze().data
	ngff_image = OMEZarrImage(
		array.rechunk((array.shape[0], 1024, 1024)),
		axes=["z", "y", "x"],
		scale=scale,
		axes_units=axes_units,
		name=Path(filename).parent.stem + img_basename
	)
	ngff_multiscales = OMEZarrMultiscale(
		image=ngff_image,
		contrast_limits=[(0, 510)],
		channel_names=["Brightfield"],
		channel_colors=["6D6D6D"]
	)
	images.append(ngff_multiscales)

	transform = Translation(
		translation=position_corner.tolist(),
		input=CoordinateSystemIdentifier(
			path=ngff_multiscales.name,
			name=ngff_multiscales.metadata.coordinateSystems[0].name
		),
		output=CoordinateSystemIdentifier(
			name="world"
		)
	)
	transforms.append(transform)

100%|██████████| 96/96 [05:07<00:00,  3.20s/it]


In [7]:
cs_world = CoordinateSystem(
	name="world",
	axes=(
		Axis(name="z", type="space", unit="micrometer"),
		Axis(name="y", type="space", unit="micrometer"),
		Axis(name="x", type="space", unit="micrometer"),
	)
)

ngff_scene = OMEZarrScene(
	images=images,
	coordinate_transformations=transforms,
	coordinate_systems=(
		cs_world,
	)
)

In [8]:
store = zarr.storage.LocalStore(r"D:\UserData\Johannes\20260512\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\zarr")

delayed = ngff_scene.to_ome_zarr(
	store,
    group="plate1/overview_image.ome.zarr",
	overwrite=True,
    compute=True,
)

Writing images: 100%|██████████| 96/96 [2:23:05<00:00, 89.43s/it]  


## Show scenes



In [25]:
viewer.layers.clear()
viewer.open("zarr/plate1/overview_image.ome.zarr", plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'A1'}, 'output': {'name': 'world'}, 'translation': [0.0028384652000001, 0.0057091052, 0.0074873634000000005]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'A10'}, 'output': {'name': 'world'}, 'translation': [0.0028384652000001, 0.0057091052, 0.0884873651]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}


[<Image layer 'A1: Brightfield' at 0x21282c5bfb0>,
 <Image layer 'A10: Brightfield' at 0x212e68107d0>]

In [17]:
viewer.layers[0].data[3].max().compute()

np.uint16(0)

## Convert SCAPE stacks

In [32]:
root = r"J:\2026_05_08_12_40_30--20260508_Plate1_SCAPE\TileScan 1"

tif_files = glob.glob(os.path.join(root, "*.ome.tif"))
images = {}
transforms = []

for tf in tqdm.tqdm(tif_files):
    img = BioImage(tf)
    slug = Path(tf).stem.split(".")[0]
    well = slug.split(" ")[0]
    name = slug.removeprefix(well + " ").replace(" ", "")
    oz_img = OMEZarrImage(
        img.dask_data.squeeze().rechunk((1, 256, 256, 256)), axes=["c", "z", "y", "x"],
        scale={"c": 1.0, "z": img.physical_pixel_sizes.Z, "y": img.physical_pixel_sizes.Y, "x": img.physical_pixel_sizes.X},
        axes_units={"z": "meter", "y": "meter", "x": "meter"},
        name=name
        )
    ms_img = OMEZarrMultiscale(oz_img, channel_names=["DAPI", "GFP"])
    col = well[0]
    row = well[1:]

    if (row, col) not in images:
        images[(row, col)] = [ms_img]
    else:
        images[(row, col)].append(ms_img)

    # parse tile metadata
    metadata_file = os.path.join(Path(tf).parent, "Metadata", f"{slug}.xlif")
    with open(metadata_file, 'rb') as f:
        content = f.read().decode('utf-8')
        metadata_dict = xmltodict.parse(content)
    tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

    translation = Translation(
        output=CoordinateSystemIdentifier(name="world"),
        input=CoordinateSystemIdentifier(path=f"{col}/{row}/{name}", name="physical"),
        translation=(0, tile_scan_info["@PosZ"], tile_scan_info["@PosY"], tile_scan_info["@PosX"])
    )
    transforms.append(translation)


100%|██████████| 2810/2810 [26:49<00:00,  1.75it/s]


In [33]:
plate = OMEZarrHCSPlate(images=images)
plate.to_ome_zarr(r"D:\UserData\Johannes\20260507\2026_05_07_10_34_04--20260507_Experiment1001\LEICA_plate1_organoids.ome.zarr", overwrite=True)

[]

## Optional: Upload tiles to OMERO

In [ ]:
host = 'omero-nfdi.uni-muenster.de'
port = 4064
user = 'JohannesSoltwedel'
group = None
secure=True

conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group, password=password)

In [32]:
for layer in viewer.layers:
	projection = layer.data[0].min(axis=0).rechunk(2560, 2560)

	# iterate over chunks and save as individual images
	for idx in np.ndindex(projection.numblocks):
		chunk = projection.blocks[idx]
		if not chunk.shape == (2560, 2560):
			continue

		tile = chunk.compute()

		if sum(tile.flatten()==1) > 10000:
			continue

		image_id = ezomero.post_image(
			conn,
			image=tile[None, None, None],
			dataset_id=251,
			image_name=f"{layer.name}_chunk_{idx}", dim_order='tczyx', across_groups=False)

		conn.setChannelNames(data_type='Image', ids=[image_id], nameDict={1: 'Brightfield'})

		unit_x = omero.model.LengthI(scale["x"]* 10e6, UnitsLength.MICROMETER)
		unit_y = omero.model.LengthI(scale["y"]* 10e6, UnitsLength.MICROMETER)

		image_object = conn.getObject("Image", image_id)
		primary_pixels_object = image_object.getPrimaryPixels()._obj
		primary_pixels_object.setPhysicalSizeX(unit_x)
		primary_pixels_object.setPhysicalSizeY(unit_y)
		conn.getUpdateService().saveObject(primary_pixels_object)
